In [1]:
# =========================================================
# UNIVERSAL HEART DISEASE RESEARCH PIPELINE
# Change DATASET_PATH and TARGET_COLUMN only
# =========================================================

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# 🔧 CHANGE ONLY THESE TWO
# =========================================================

DATASET_PATH = "datasets/heart_scaled.csv"
TARGET_COLUMN = "HeartDisease"

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(DATASET_PATH)

# Basic cleaning assumption:
df = df.dropna()

X_df = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN].values

# Convert categorical if any
X_df = pd.get_dummies(X_df, drop_first=True)

print("\nDataset Loaded Successfully")
print("Shape:", X_df.shape)
print("Class Distribution:")
print(pd.Series(y).value_counts(normalize=True))

# =========================================================
# CROSS VALIDATION SETUP
# =========================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_auc = []
stack_auc = []

baseline_brier = []
stack_brier = []

# =========================================================
# CV LOOP
# =========================================================

for fold, (train_idx, test_idx) in enumerate(skf.split(X_df, y)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train = X_df.iloc[train_idx]
    X_test  = X_df.iloc[test_idx]

    y_train = y[train_idx]
    y_test  = y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # ----------------------------
    # BASELINE ET
    # ----------------------------
    et = ExtraTreesClassifier(n_estimators=200, random_state=42)
    et.fit(X_train_scaled, y_train)

    et_test_proba = et.predict_proba(X_test_scaled)[:, 1]

    auc_base = roc_auc_score(y_test, et_test_proba)
    brier_base = brier_score_loss(y_test, et_test_proba)

    baseline_auc.append(auc_base)
    baseline_brier.append(brier_base)

    print("Baseline ET AUC:", round(auc_base,4))

    # ----------------------------
    # ET + NB STACKING
    # ----------------------------
    nb = GaussianNB()
    nb.fit(X_train_scaled, y_train)

    et_train_proba = et.predict_proba(X_train_scaled)[:, 1]
    nb_train_proba = nb.predict_proba(X_train_scaled)[:, 1]

    nb_test_proba = nb.predict_proba(X_test_scaled)[:, 1]

    meta_train = np.column_stack([et_train_proba, nb_train_proba])
    meta_test  = np.column_stack([et_test_proba, nb_test_proba])

    meta_lr = LogisticRegression(max_iter=1000)
    meta_lr.fit(meta_train, y_train)

    stack_test_proba = meta_lr.predict_proba(meta_test)[:, 1]

    auc_stack = roc_auc_score(y_test, stack_test_proba)
    brier_stack = brier_score_loss(y_test, stack_test_proba)

    stack_auc.append(auc_stack)
    stack_brier.append(brier_stack)

    print("ET+NB AUC:", round(auc_stack,4))

# =========================================================
# FINAL RESULTS
# =========================================================

print("\n==============================")
print("FINAL CROSS-VALIDATED RESULTS")
print("==============================")

print(f"Baseline ET Mean AUC: {np.mean(baseline_auc):.4f} | Std: {np.std(baseline_auc):.4f}")
print(f"ET+NB Mean AUC: {np.mean(stack_auc):.4f} | Std: {np.std(stack_auc):.4f}")

print("\nBaseline ET Mean Brier:", round(np.mean(baseline_brier),4))
print("ET+NB Mean Brier:", round(np.mean(stack_brier),4))

# =========================================================
# STATISTICAL SIGNIFICANCE
# =========================================================

if not np.allclose(baseline_auc, stack_auc):
    t_stat, p_value = ttest_rel(stack_auc, baseline_auc)
    print("\nPaired t-test (AUC) p-value:", round(p_value,6))
else:
    print("\nModels identical across folds. No statistical difference.")

# =========================================================
# ROBUSTNESS TEST (Noise Injection)
# =========================================================

noise_levels = [0.0, 0.1, 0.2]

print("\n===== ROBUSTNESS UNDER NOISE =====")

for noise in noise_levels:

    et_scores = []
    stack_scores = []

    for train_idx, test_idx in skf.split(X_df, y):

        X_train = X_df.iloc[train_idx]
        X_test  = X_df.iloc[test_idx]

        y_train = y[train_idx]
        y_test  = y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        X_test_noisy = X_test_scaled + np.random.normal(0, noise, X_test_scaled.shape)

        et = ExtraTreesClassifier(n_estimators=200, random_state=42)
        et.fit(X_train_scaled, y_train)
        et_proba = et.predict_proba(X_test_noisy)[:, 1]
        et_scores.append(roc_auc_score(y_test, et_proba))

        nb = GaussianNB()
        nb.fit(X_train_scaled, y_train)

        et_train_proba = et.predict_proba(X_train_scaled)[:, 1]
        nb_train_proba = nb.predict_proba(X_train_scaled)[:, 1]

        nb_test_proba = nb.predict_proba(X_test_noisy)[:, 1]

        meta_train = np.column_stack([et_train_proba, nb_train_proba])
        meta_test  = np.column_stack([et_proba, nb_test_proba])

        meta_lr = LogisticRegression(max_iter=1000)
        meta_lr.fit(meta_train, y_train)

        stack_proba = meta_lr.predict_proba(meta_test)[:, 1]
        stack_scores.append(roc_auc_score(y_test, stack_proba))

    print(f"\nNoise Level: {noise}")
    print("ET Mean AUC:", round(np.mean(et_scores),4))
    print("ET+NB Mean AUC:", round(np.mean(stack_scores),4))


Dataset Loaded Successfully
Shape: (1190, 11)
Class Distribution:
1.0    0.528571
0.0    0.471429
Name: proportion, dtype: float64

========== Fold 1 ==========
Baseline ET AUC: 0.9664
ET+NB AUC: 0.9597

========== Fold 2 ==========
Baseline ET AUC: 0.9796
ET+NB AUC: 0.9764

========== Fold 3 ==========
Baseline ET AUC: 0.9789
ET+NB AUC: 0.9751

========== Fold 4 ==========
Baseline ET AUC: 0.9782
ET+NB AUC: 0.9751

========== Fold 5 ==========
Baseline ET AUC: 0.9798
ET+NB AUC: 0.9768

FINAL CROSS-VALIDATED RESULTS
Baseline ET Mean AUC: 0.9766 | Std: 0.0051
ET+NB Mean AUC: 0.9726 | Std: 0.0065

Baseline ET Mean Brier: 0.0553
ET+NB Mean Brier: 0.0545

Paired t-test (AUC) p-value: 0.005115

===== ROBUSTNESS UNDER NOISE =====

Noise Level: 0.0
ET Mean AUC: 0.9766
ET+NB Mean AUC: 0.9726

Noise Level: 0.1
ET Mean AUC: 0.9692
ET+NB Mean AUC: 0.9667

Noise Level: 0.2
ET Mean AUC: 0.9638
ET+NB Mean AUC: 0.9604


2nd Pipeline

In [5]:
# =========================================================
# UNIVERSAL HEART RESEARCH PIPELINE V3
# Baseline ET vs ET+NB vs Calibrated ET+NB
# Change DATASET_PATH and TARGET_COLUMN only
# =========================================================

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# 🔧 CHANGE ONLY THESE
# =========================================================

DATASET_PATH = "datasets/framingham_cleaned.csv"
TARGET_COLUMN = "HeartDisease"

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(DATASET_PATH)
df = df.dropna()

X_df = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN].values

# Convert categorical variables automatically
X_df = pd.get_dummies(X_df, drop_first=True)

print("\nDataset Loaded Successfully")
print("Shape:", X_df.shape)
print("Class Distribution:")
print(pd.Series(y).value_counts(normalize=True))

# =========================================================
# CROSS VALIDATION SETUP
# =========================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_auc = {
    "ET": [],
    "STACK": [],
    "STACK_CALIBRATED": []
}

results_brier = {
    "ET": [],
    "STACK": [],
    "STACK_CALIBRATED": []
}

# =========================================================
# CV LOOP
# =========================================================

for fold, (train_idx, test_idx) in enumerate(skf.split(X_df, y)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train = X_df.iloc[train_idx]
    X_test  = X_df.iloc[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # =============================
    # 1️⃣ Baseline Extra Trees
    # =============================

    et = ExtraTreesClassifier(n_estimators=200, random_state=42)
    et.fit(X_train_scaled, y_train)

    et_test_proba = et.predict_proba(X_test_scaled)[:, 1]

    results_auc["ET"].append(roc_auc_score(y_test, et_test_proba))
    results_brier["ET"].append(brier_score_loss(y_test, et_test_proba))

    print("Baseline ET AUC:", round(results_auc["ET"][-1],4))

    # =============================
    # 2️⃣ ET + NB Stacking
    # =============================

    nb = GaussianNB()
    nb.fit(X_train_scaled, y_train)

    et_train_proba = et.predict_proba(X_train_scaled)[:, 1]
    nb_train_proba = nb.predict_proba(X_train_scaled)[:, 1]

    et_test_proba = et.predict_proba(X_test_scaled)[:, 1]
    nb_test_proba = nb.predict_proba(X_test_scaled)[:, 1]

    meta_train = np.column_stack([et_train_proba, nb_train_proba])
    meta_test  = np.column_stack([et_test_proba, nb_test_proba])

    meta_lr = LogisticRegression(max_iter=1000)
    meta_lr.fit(meta_train, y_train)

    stack_test_proba = meta_lr.predict_proba(meta_test)[:, 1]

    results_auc["STACK"].append(roc_auc_score(y_test, stack_test_proba))
    results_brier["STACK"].append(brier_score_loss(y_test, stack_test_proba))

    print("ET+NB AUC:", round(results_auc["STACK"][-1],4))

    # =============================
    # 3️⃣ ET + NB + Platt Calibration
    # =============================

    # Split meta_train into model-train and calibration split
    split_index = int(0.8 * len(meta_train))

    meta_train_part = meta_train[:split_index]
    meta_cal_part   = meta_train[split_index:]

    y_train_part = y_train[:split_index]
    y_cal_part   = y_train[split_index:]

    # Train meta model
    meta_lr_cal = LogisticRegression(max_iter=1000)
    meta_lr_cal.fit(meta_train_part, y_train_part)

    # Calibration probabilities
    cal_proba = meta_lr_cal.predict_proba(meta_cal_part)[:,1]

    # Platt scaling
    platt = LogisticRegression()
    platt.fit(cal_proba.reshape(-1,1), y_cal_part)

    # Apply to test
    raw_test_proba = meta_lr_cal.predict_proba(meta_test)[:,1]
    calibrated_proba = platt.predict_proba(raw_test_proba.reshape(-1,1))[:,1]

    results_auc["STACK_CALIBRATED"].append(
        roc_auc_score(y_test, calibrated_proba)
    )

    results_brier["STACK_CALIBRATED"].append(
        brier_score_loss(y_test, calibrated_proba)
    )

    print("ET+NB Calibrated AUC:", round(results_auc["STACK_CALIBRATED"][-1],4))

# =========================================================
# FINAL RESULTS
# =========================================================

print("\n==============================")
print("FINAL CROSS-VALIDATED RESULTS")
print("==============================")

for model in results_auc:
    print(f"\n{model}")
    print("Mean AUC:", round(np.mean(results_auc[model]),4),
          "| Std:", round(np.std(results_auc[model]),4))
    print("Mean Brier:", round(np.mean(results_brier[model]),4))

# =========================================================
# STATISTICAL TEST
# =========================================================

if not np.allclose(results_auc["STACK"], results_auc["ET"]):
    t_stat, p_value = ttest_rel(results_auc["STACK"], results_auc["ET"])
    print("\nPaired t-test (STACK vs ET) p-value:", round(p_value,6))
else:
    print("\nNo difference between ET and STACK.")



Dataset Loaded Successfully
Shape: (4240, 15)
Class Distribution:
0    0.848113
1    0.151887
Name: proportion, dtype: float64

========== Fold 1 ==========
Baseline ET AUC: 0.6989
ET+NB AUC: 0.7009
ET+NB Calibrated AUC: 0.7009

========== Fold 2 ==========
Baseline ET AUC: 0.704
ET+NB AUC: 0.7089
ET+NB Calibrated AUC: 0.7089

========== Fold 3 ==========
Baseline ET AUC: 0.706
ET+NB AUC: 0.7089
ET+NB Calibrated AUC: 0.7089

========== Fold 4 ==========
Baseline ET AUC: 0.7151
ET+NB AUC: 0.7186
ET+NB Calibrated AUC: 0.7187

========== Fold 5 ==========
Baseline ET AUC: 0.65
ET+NB AUC: 0.6542
ET+NB Calibrated AUC: 0.6541

FINAL CROSS-VALIDATED RESULTS

ET
Mean AUC: 0.6948 | Std: 0.023
Mean Brier: 0.1204

STACK
Mean AUC: 0.6983 | Std: 0.0227
Mean Brier: 0.1355

STACK_CALIBRATED
Mean AUC: 0.6983 | Std: 0.0228
Mean Brier: 0.142

Paired t-test (STACK vs ET) p-value: 0.002278
